In [ ]:
# Graph-based Spatial Clustering

## Original Applied Data Science Project, 2025

University of Manchester

Industry collaboration: Geodesignhub

## Research Objective

To group heterogeneous citizen-drawn spatial geometries
using spatial adjacency, Jaccard similarity and graph
connectivity.

## My Contribution

Methodological design of the topological clustering
approach, AI-assisted implementation, spatial analysis
and research documentation.

## Notebook Status

This notebook preserves the original analytical logic.
Known methodological and implementation limitations
are documented separately.

The original industry dataset is not publicly distributed.

In [ ]:
# Import Library & Read Data
import geopandas as gpd
import pandas as pd
import colorsys
import numpy as np
from collections import defaultdict
import networkx as nx
import json
from bokeh.io import output_file, save
from bokeh.plotting import figure
from bokeh.palettes import linear_palette
from matplotlib.colors import to_rgb, to_hex

In [ ]:
# Calculate area, perimeter, shape index, center point only for legal geometry
gdf.loc[gdf.is_valid_geom, "area"] = gdf.loc[gdf.is_valid_geom, "geometry"].area
gdf.loc[gdf.is_valid_geom, "perimeter"] = gdf.loc[gdf.is_valid_geom, "geometry"].length
gdf.loc[gdf.is_valid_geom, "shape_index"] = (
    gdf.loc[gdf.is_valid_geom, "perimeter"] / (4 * np.sqrt(gdf.loc[gdf.is_valid_geom, "area"]))
)
gdf.loc[gdf.is_valid_geom, "centroid"] = gdf.loc[gdf.is_valid_geom, "geometry"].centroid
gdf.loc[gdf.is_valid_geom, "x"] = gdf.loc[gdf.is_valid_geom, "centroid"].x
gdf.loc[gdf.is_valid_geom, "y"] = gdf.loc[gdf.is_valid_geom, "centroid"].y

# Category field cleaning (keep &, do not remove symbols)
gdf["main_category"] = gdf["Category"].fillna("Unknown").str.strip()

In [ ]:
# Create location_group (only process legal geometry)
buffer_radius = 20
gdf_valid = gdf[gdf.is_valid_geom].copy()
gdf_valid["buffer"] = gdf_valid.geometry.buffer(buffer_radius)
joined = gpd.sjoin(
    gdf_valid[["id", "buffer"]].set_geometry("buffer"),
    gdf_valid[["id", "geometry"]],
    how="left", predicate="intersects"
)
joined = joined[joined["id_left"] != joined["id_right"]]

def jaccard_similarity(geom1, geom2):
    try:
        inter = geom1.intersection(geom2).area
        union = geom1.union(geom2).area
        return inter / union if union > 0 else 0
    except:
        return 0

G = nx.Graph()
for _, row in joined.iterrows():
    id1 = row["id_left"]
    id2 = row["id_right"]
    poly1 = gdf_valid.loc[gdf_valid["id"] == id1].geometry.values[0]
    poly2 = gdf_valid.loc[gdf_valid["id"] == id2].geometry.values[0]
    sim = jaccard_similarity(poly1, poly2)
    if sim >= 0.3:
        G.add_edge(id1, id2)

id_to_group = {}
for i, component in enumerate(nx.connected_components(G)):
    for geom_id in component:
        id_to_group[geom_id] = i
gdf_valid["location_group"] = gdf_valid["id"].map(id_to_group)

In [ ]:
# No. geo_subclass (valid and invalid separated)
geo_counter = defaultdict(int)
gdf["geo_subclass"] = None  
gdf["main_category"] = gdf["Category"] 
gdf_valid["main_category"] = gdf["main_category"]

category_prefix = {
    "🚧 Fences & Barriers": "F",
    "🌳 Planting": "PL",
    "🚗 Parking": "P",
    "🚴 Bike use": "B",
    "🚻 Restroom": "R",
    "🎡 Recreation Area": "RE",
    "🚶 Pedestrian Use": "PE",
    "🧭 Wayfinding&Signage": "W",
    "💧 Water Services": "WS"
}

In [ ]:
# Assign geo_subclass with location_group of valid geometry
for cat in gdf["main_category"].unique():
    clean_cat = cat.strip()
    prefix = category_prefix.get(clean_cat, clean_cat[:2].upper())

    # Valid + grouped parts
    sub_valid = gdf_valid[gdf_valid["main_category"] == cat]
    for group in sorted(sub_valid["location_group"].dropna().unique()):
        geo_counter[prefix] += 1
        label = f"{prefix}{geo_counter[prefix]:02d}"

        ids_in_group = sub_valid[sub_valid["location_group"] == group]["id"]
        mask = (gdf["id"].isin(ids_in_group))
        gdf.loc[mask, "geo_subclass"] = label

    # Invalid geometry part
    invalid_mask = (gdf["main_category"] == cat) & (~gdf.is_valid_geom)
    if invalid_mask.any():
        geo_counter[prefix] += 1
        label = f"{prefix}{geo_counter[prefix]:02d}"
        gdf.loc[invalid_mask, "geo_subclass"] = label
        gdf.loc[invalid_mask, "location_group"] = -1

In [ ]:
# Handling of unnumbered valid figures
unassigned_valid = gdf[gdf["geo_subclass"].isna() & gdf["is_valid_geom"]]

# Assign numbers to valid unnumbered shapes
for _, row in unassigned_valid.iterrows():
    cat = row["main_category"]
    clean_cat = cat.strip()
    prefix = category_prefix.get(clean_cat, clean_cat[:2].upper())
    
    geo_counter[prefix] += 1
    label = f"{prefix}{geo_counter[prefix]:02d}"
    
    # update
    gdf.loc[gdf["id"] == row["id"], "geo_subclass"] = label

# print
print(gdf["geo_subclass"].value_counts())
unassigned = gdf[gdf.is_valid_geom & gdf["geo_subclass"].isna()]
print(f"The number of valid geometries that have not yet been numbered：{len(unassigned)}")
print("success：")
print(gdf["geo_subclass"].value_counts())

In [ ]:
# Difference detection (valid only)
# make sure the indexes of gdf_valid are continuous
gdf_valid = gdf_valid.reset_index(drop=True)

# filter
gdf_valid = gdf[gdf.geometry.notnull() & gdf.geometry.is_valid].copy()

# define
def compute_differences(df_group):
    ids = df_group["id"].values
    geometries = df_group.geometry.values
    flags = [False] * len(df_group)
    for i in range(len(ids)):
        for j in range(i + 1, len(ids)):
            c_dist = geometries[i].centroid.distance(geometries[j].centroid)
            a_diff = abs(geometries[i].area - geometries[j].area) / max(geometries[i].area, geometries[j].area)
            if c_dist < 10 and a_diff > 0.3:
                flags[i] = flags[j] = True
    return flags

# Group geo_subclass by geo_subclass and mark the differences
flags_all = []
for subclass in gdf_valid["geo_subclass"].dropna().unique():
    group = gdf_valid[gdf_valid["geo_subclass"] == subclass].copy()
    flags = compute_differences(group)
    flags_all.extend(flags)

if len(flags_all) != len(gdf_valid):
    gdf_valid["wrong"] = [False] * len(gdf_valid)
else:
    gdf_valid["wrong"] = flags_all


#  Merge the difference markers into the original gdf (keep invalid data)
gdf = gdf.drop(columns=["wrong"], errors="ignore")  
gdf = gdf.merge(gdf_valid[["id", "wrong"]], on="id", how="left")

In [ ]:
# Create hue variant function (brightness adjustment based on HLS model)
def generate_shades(base_hex, n):
    base_rgb = to_rgb(base_hex)  # 0-1 RGB
    h, l, s = colorsys.rgb_to_hls(*base_rgb)

    # Ensure that there is a distribution of brightness (from light to dark) to avoid blackening.
    lightness_range = [0.85 - i * 0.5 / max(n-1, 1) for i in range(n)]
    shades = [to_hex(colorsys.hls_to_rgb(h, l_new, s)) for l_new in lightness_range]
    return shades

# Main category color (fixed)
category_base_colors = {
    "🚧 Fences & Barriers": "#FFFFFF",      # white
    "🌳 Planting": "#1a9641",               # green
    "🚗 Parking": "#7b3294",                # purple
    "🚴 Bike use": "#d7191c",               # red
    "🚻 Restroom": "#fdae61",               # yellow
    "🎡 Recreation Area": "#8c510a",        # brown
    "🚶 Pedestrian Use": "#636363",         # grey
    "🧭 Wayfinding&Signage": "#f46d43",     # orange
    "💧 Water Services": "#2b83ba"          # blue
}

# update gdf_valid
gdf_valid = gdf[gdf.is_valid_geom & gdf["geo_subclass"].notna()].copy()

# Creating color maps in groups
geo_color_map = {}
grouped = gdf_valid.groupby("main_category")["geo_subclass"].unique()

for main_cat, subclasses in grouped.items():
    base_color = category_base_colors.get(main_cat, "#cccccc")  # fallback 
    n = len(subclasses)
    shades = generate_shades(base_color, n)
    for sub, shade in zip(sorted(subclasses), shades):
        geo_color_map[sub] = shade

gdf_valid["fillcolor"] = gdf_valid["geo_subclass"].map(geo_color_map)
gdf_valid["fillcolor"] = gdf_valid["fillcolor"].astype(str)

In [ ]:
#  export excel
output_rows = []
for subclass in sorted(gdf["geo_subclass"].dropna().unique()):
    group = gdf[gdf["geo_subclass"] == subclass]
    for i, (_, row) in enumerate(group.iterrows()):
        output_rows.append({
            "category": row["main_category"] if i == 0 else "",
            "sub": subclass if i == 0 else "",
            "response": row["Response"],
            "area (㎡)": round(row["area"], 2),
            "shape_index": round(row["shape_index"], 2),
            "wrong": row["wrong"],
             "Open Location Code": row.get("Open Location Code", "")        
        })

output_df = pd.DataFrame(output_rows)
output_df.to_excel("output_location_differences2.xlsx", index=False)